# 프랜차이즈 발표 실습

이 노트북은 **이번 발표 준비용으로 추가한 연결·시연 코드**입니다. 이전 원본 코드를 바꾸지 않았습니다. `reference_code/analysis.py`의 실제 분석 함수를 사용합니다.

압축을 풀고 그 폴더를 VS Code에서 연 다음 이 노트북을 위에서 아래로 실행하세요. 인터넷 없이 실행합니다. 기존 데이터를 새로 수집했다는 뜻은 아닙니다. 자료에 있는 숫자를 그대로 사용하므로 시장 데이터의 정확성을 자동 보증하지 않습니다.

## 1. 도구를 가져오기
`import`는 이미 만들어진 도구를 사용하겠다는 뜻입니다. pandas는 표 계산, json은 JSON 읽기, Path는 파일 경로 처리에 사용합니다. `sys.path`에는 기존 분석 코드가 있는 폴더를 추가합니다.

In [ ]:
from pathlib import Path
import json
import hashlib
import sys
import pandas as pd

ROOT = Path.cwd()
assert (ROOT / "data/json").is_dir(), "압축을 푼 발표 패키지 폴더에서 노트북을 실행하세요."
sys.path.insert(0, str(ROOT / "reference_code"))
from analysis import industry_average, compare_sales, aggregate, budget_profit, year_on_year


## 2. 수집해 둔 공정위 표 확인
이 파일은 과거 수집 결과입니다. 지금 사이트를 다시 접속한 결과가 아닙니다. `disclosure_2025_tables`는2025년 등록 공시에 있는 표이며, 이 예시의 실제 매출 연도는2024년입니다.

In [ ]:
evidence = json.loads((ROOT / "data/evidence/CU.json").read_text(encoding="utf-8"))
revenue_rows = evidence["disclosure_2025_tables"][8]["rows"]
assert revenue_rows[0][-1] == "2024년"
cu_source_table = pd.DataFrame(revenue_rows[2:], columns=["지역", "가맹점수", "평균매출액_천원", "면적당매출_천원"])
cu_source_table.head()


## 3. 문자열과 단위 정리
천원 표기633,837에서 쉼표를 없애 숫자로 바꾸고1000을 곱합니다. 이는 연평균 점포 매출이며 본사 매출이나 전국 총매출이 아닙니다.

In [ ]:
national_row = next(row for row in revenue_rows[2:] if row[0] == "전체")
annual_revenue = int(national_row[2].replace(",", "")) * 1000
assert annual_revenue == 633837000
annual_revenue


## 4. 마지막에 만든 엑셀 보존용 JSON 읽기
마지막 JSON은 `cells`가 아니라 `index`, `columns`, `data` 구조입니다. 기존 분석 로더에 그대로 넣지 않고 아래에서 연결합니다. 원본 파일이 변하지 않았는지도 해시로 확인합니다.

In [ ]:
json_files = sorted((ROOT / "data/json").glob("프랜차이즈_*.json"))
assert len(json_files) == 4
for path in json_files:
    obj = json.loads(path.read_text(encoding="utf-8"))
    excel_path = ROOT / "data/excel" / obj["source_filename"]
    assert hashlib.sha256(excel_path.read_bytes()).hexdigest() == obj["source_sha256"]

cu_json_file = next(p for p in json_files if "_편의점_" in p.name)
cu_json = json.loads(cu_json_file.read_text(encoding="utf-8"))
sheet = next(s for s in cu_json["sheets"] if s["name"] == "05_매출비교")
excel_view = pd.DataFrame(sheet["data"], index=sheet["index"], columns=sheet["columns"])
excel_view.loc[2:6, "A":"F"]


## 5. 계산하기 쉬운 행으로 연결
한 행을 연도·업종·브랜드·지역·값으로 만듭니다. 매출과 점포 수는 다른 표로 분리합니다. 기존 원본 값은 바꾸지 않습니다. 원본의 `0`도 보존하고, 숫자가 아닌 `미공계` 같은 텍스트는 숫자로 추정하지 않습니다. 이 반복문은 발표용으로 새로 작성한 연결 코드입니다.

In [ ]:
sales_records = []
store_records = []

for path in json_files:
    obj = json.loads(path.read_text(encoding="utf-8"))
    industry = obj["source_filename"].split("_")[1]
    sheets = {s["name"]: s for s in obj["sheets"]}
    configs = [
        ("05_매출비교", sales_records, [(2, "전국"), (3, "서울"), (4, "경기")]),
        ("03_지역집계", store_records, [(4, "전국"), (2, "서울"), (3, "경기")])
    ]
    for sheet_name, output, region_columns in configs:
        for row in sheets[sheet_name]["data"][2:]:
            if len(row) < 6 or row[1] != industry:
                continue
            if str(row[5]) not in ("2024", "2025"):
                continue
            for col, region in region_columns:
                value = row[col]
                if isinstance(value, (int, float)) and not isinstance(value, bool):
                    output.append({
                        "year": int(row[5]), "industry": industry,
                        "brand": row[0], "region": region, "value": value
                    })

sales = pd.DataFrame(sales_records)
stores = pd.DataFrame(store_records)
keys = ["year", "industry", "brand", "region"]
assert not sales.duplicated(keys).any()
assert sales.year.isin([2024, 2025]).all()
sales.head()


## 6. 기존 분석 함수 실행: 업종별 평균
`groupby`는 같은 업종끼리 묶고, `mean`은 평균을 냅니다. 이 결과는 입력 파일의 선정 브랜드 단순평균입니다. 시장 전체의 공식 평균이 아닙니다. `brand_count`를 함께 보세요.

In [ ]:
average_table = industry_average(sales, 2024, "전국")
average_table


## 7. 기존 함수 실행: 같은 연도·지역의 매출 순위
매출이 큰 브랜드가1위입니다. 평균 대비 차이도 같이 표시합니다. 다른 연도의 값을 한 순위표에 섞지 않습니다.

In [ ]:
ranking = compare_sales(sales, 2025, "서울")
ranking[["year", "industry", "brand", "region", "value", "industry_rank", "difference_from_mean"]]


## 8. 지역·업종 집계와 전년 비교
전국과 서울·경기를 합치지 않습니다. 전년 비교는 같은 브랜드의 두 연도가 모두 있어야 숫자가 나옵니다. NaN은 관측값이 없는 상태이지 매출0이 아닙니다.

In [ ]:
summary = aggregate(sales, stores, 2024, "서울", by="industry")
summary


In [ ]:
comparison = year_on_year(sales, "전국")
comparison[["industry", "brand", "value_2024", "value_2025", "change", "growth_rate"]]


## 9. CU 비용·운영이익 직접 계산
아래 비율은 원본00_가정에 있는 편의점 기본 비율입니다. 실제 지출을 수집한 값이 아닙니다. 동일한83% 비용 비율을 사용하면17% 이익률이 나오는 구조입니다.

In [ ]:
costs = {
    "material": annual_revenue * 0.65,
    "labor": annual_revenue * 0.08,
    "rent": annual_revenue * 0.05,
    "other": annual_revenue * 0.05
}
profit_result = budget_profit(annual_revenue, costs)
assert profit_result["annual_total_cost"] == 526084710
assert profit_result["operating_profit"] == 107752290
assert profit_result["margin"] == 0.17
pd.DataFrame([{
    "연매출": profit_result["annual_revenue"],
    "운영비합계": profit_result["annual_total_cost"],
    "예상운영이익": profit_result["operating_profit"],
    "예상이익률": profit_result["margin"]
}])


## 10. 비용을 바꾸면 이익이 바뀌는지 확인
아래는 기능 검사용 입력 변화입니다. 실제 임차료 조사값이100만원 늘었다는 뜻이 아닙니다. 원본 엑셀을 수정하지 않습니다.

In [ ]:
changed_costs = costs.copy()
changed_costs["rent"] += 1000000
changed_result = budget_profit(annual_revenue, changed_costs)
assert profit_result["operating_profit"] - changed_result["operating_profit"] == 1000000
pd.DataFrame([
    {"조건": "원본 가정", "운영이익": profit_result["operating_profit"]},
    {"조건": "기능검사: 임차료100만원 추가", "운영이익": changed_result["operating_profit"]}
])


## 11. 수식과 저장 계산값 읽기
openpyxl은 수식을 계산하는 도구가 아닙니다. `data_only=False`는 수식, `True`는 파일에 저장된 결과를 읽습니다. 아래에서 같은CU셀의 두 모습을 확인합니다.

In [ ]:
import openpyxl
cu_excel = ROOT / "data/excel" / cu_json["source_filename"]
formulas_book = openpyxl.load_workbook(cu_excel, data_only=False)
values_book = openpyxl.load_workbook(cu_excel, data_only=True)
checks = []
for address in ["C3", "E3", "F3", "G3", "H3", "I3", "J3", "K3"]:
    checks.append({
        "셀": address,
        "수식": formulas_book["06_운영이익"][address].value,
        "저장계산결과": values_book["06_운영이익"][address].value
    })
pd.DataFrame(checks)


## 12. 실제 변환 함수로 JSON 저장 시연
마지막 변환 때 사용했던 `convert_excel.py`의 함수를 호출합니다. 결과는 `demo_output`에 별도로 저장합니다. 입력 엑셀과 기존JSON을 덮어쓰지 않습니다.

In [ ]:
from convert_excel import convert
output_dir = ROOT / "demo_output"
output_dir.mkdir(exist_ok=True)
conversion_report = convert(cu_excel, output_dir / "CU_demo.json")
assert conversion_report["all_cells_verified"]
conversion_report


## 13. 분석 결과 JSON 저장
파일 변환 결과와 계산 결과는 다른 산출물입니다. 아래는CU 시뮬레이션 계산 결과만 별도 저장하는 예시입니다. 한글은UTF-8로 저장하고NaN/Infinity는 허용하지 않습니다.

In [ ]:
(output_dir / "CU_profit_demo.json").write_text(
    json.dumps(profit_result, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8"
)
assert json.loads((output_dir / "CU_profit_demo.json").read_text(encoding="utf-8"))["operating_profit"] == 107752290
"수집자료 확인, 분석함수 실행, 계산 검증, JSON 저장 완료"


## 발표 마무리
원본 숫자 자체가 정확한지 확인하는 검증과, 수식·파일변환이 제대로 되었는지 확인하는 검증은 다릅니다. 이 실습의 통과는 사이트의 모든 값이 정확하다는 보증이 아닙니다. API 서버·온라인 스크래핑은 이 노트북에서 실행하지 않습니다.